# 🏆 BTK Datathon 2026 — v8.1 (TabPFN-3 + Memory-Safe NLP)

**Yarışma Metriği:** MSE  
**Donanım:** NVIDIA A100 GPU  
**Felsefe:** *Frontier model (TabPFN-3, Mayıs 2026) + bellek güvenli pipeline*

## v8 → v8.1 Düzeltmeleri

| Sorun (v8) | Düzeltme (v8.1) |
|------------|------------------|
| BERT + TF-IDF aynı hücrede → kernel ölümü | **3 ayrı hücreye ayrıldı** |
| `char_wb (3,5)` Türkçe için ağır | **`char_wb (3,4)`** |
| `max_features=3000` (char) | **1500** |
| `min_df=3` (zayıf filtre) | **min_df=10** |
| Bellek temizleme yetersiz | **Her büyük adımdan sonra gc.collect()** |

## TabPFN-3 Hakkında (Doğru Bilgi)

**Prior Labs** tarafından **12 Mayıs 2026'da** yayınlanan tablo verisi foundation model'i.

### v3'ün Yenilikleri (v2 → v3)
- 1M satıra kadar destek (v2: 10K)
- 10x daha hızlı inference
- **Tabular-text data** desteği ⭐ (mentor_feedback_text için)
- TabPFN-3-Plus (Thinking) AutoGluon 1.5'i geçiyor
- SAP tarafından Mayıs 2026'da 1+ milyar EUR ile satın alma duyurusu

### Bizim Veri için Uygunluğu
- ✅ 10000 satır (TabPFN-3 max: 1M)
- ✅ ~250 feature (max: 2000 @ 100K satır için)
- ✅ Mixed types + tabular-text

## ⚠️ ÖNEMLİ: TabPFN HuggingFace Auth (TEK SEFERLİK)

TabPFN-v3 modeli HuggingFace üzerinde **gated repo** olarak yayınlandı. Çalıştırmadan önce:

### Adımlar:
1. **HF hesabı yoksa aç:** https://huggingface.co/join
2. **Lisansı kabul et:** https://huggingface.co/Prior-Labs/tabpfn_3 → "Agree and access repository"
3. **Token al:** https://huggingface.co/settings/tokens → "Create new token" → **Read** yetkisiyle
4. **Aşağıdaki hücrede token'ını gir** (sadece bu run için, kaydedilmez)

**Eğer auth yapmazsan:** Notebook otomatik fallback yapar (XGBoost ile devam eder, biraz daha düşük skor).

In [ ]:
# ── Paket kurulumu ──────────────────────────────────────────────────────────
!pip install tabpfn flaml sentence-transformers -q 2>&1 | tail -3
!pip install xgboost lightgbm catboost optuna -q 2>&1 | tail -1

In [ ]:
# ── HuggingFace Auth ─────────────────────────────────────────────────────────
# Token'ını buraya yapıştır (https://huggingface.co/settings/tokens)
# Eğer boş bırakırsan → TabPFN devre dışı, XGBoost fallback aktif

HF_TOKEN = ''  # 'hf_xxxxxxxxxxxx' formatında

if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('✓ HuggingFace authentication başarılı')
    except Exception as e:
        print(f'⚠ HF auth hatası: {str(e)[:100]}')
        print('  TabPFN devre dışı kalacak, XGBoost fallback aktif')
else:
    print('⚠ HF_TOKEN boş — TabPFN devre dışı, XGBoost fallback aktif')
    print('  TabPFN kullanmak için token gir ve hücreyi tekrar çalıştır')

In [ ]:
import pandas as pd
import numpy as np
import warnings, gc, time
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from flaml import tune as flaml_tune

SEED          = 42
N_FOLDS       = 10
N_SEEDS       = 3
HOLDOUT_RATIO = 0.2
TARGET_COL    = 'career_success_score'

np.random.seed(SEED)
print('✓ Kütüphaneler hazır')

In [ ]:
# ── GPU + TabPFN Hazırlık ────────────────────────────────────────────────────
GPU_AVAILABLE = False; LGB_GPU = False; CAT_GPU = False
BERT_DEVICE = 'cpu'; TABPFN_DEVICE = 'cpu'; TABPFN_OK = False

tx = np.random.rand(100,5).astype(np.float32); ty = np.random.rand(100).astype(np.float32)

try:
    m = xgb.XGBRegressor(n_estimators=10, device='cuda', tree_method='hist'); m.fit(tx, ty)
    GPU_AVAILABLE = True; print('✓ XGBoost GPU')
except: print('✗ XGBoost CPU')

try:
    m = lgb.LGBMRegressor(n_estimators=10, device='gpu', verbose=-1); m.fit(tx, ty)
    LGB_GPU = True; print('✓ LightGBM GPU')
except: print('✗ LightGBM CPU')

try:
    m = CatBoostRegressor(iterations=10, task_type='GPU', devices='0', verbose=0); m.fit(tx, ty)
    CAT_GPU = True; print('✓ CatBoost GPU')
except: print('✗ CatBoost CPU')

try:
    import torch
    if torch.cuda.is_available():
        BERT_DEVICE = 'cuda'; TABPFN_DEVICE = 'cuda'
        print(f'✓ PyTorch CUDA: {torch.cuda.get_device_name(0)}')
except: print('✗ PyTorch yok')

# TabPFN testi (auth'lanmışsa)
if HF_TOKEN:
    try:
        from tabpfn import TabPFNRegressor
        # Hızlı test
        print('TabPFN-v3 testi...')
        tp_test = TabPFNRegressor(device=TABPFN_DEVICE, n_estimators=2, ignore_pretraining_limits=True)
        tp_test.fit(tx, ty)
        _ = tp_test.predict(tx[:10])
        TABPFN_OK = True
        print('✓ TabPFN-v3 çalışıyor — MERKEZ MODEL OLARAK KULLANILACAK')
    except Exception as e:
        print(f'✗ TabPFN hatası: {str(e)[:120]}')
        print('  XGBoost fallback aktif')
        TABPFN_OK = False

xgb_gpu = lambda: {'device':'cuda','tree_method':'hist'} if GPU_AVAILABLE else {'tree_method':'hist','n_jobs':-1}
lgb_gpu = lambda: {'device':'gpu'} if LGB_GPU else {'n_jobs':-1}
cat_gpu = lambda: {'task_type':'GPU','devices':'0'} if CAT_GPU else {}

print(f'\n{"═"*50}')
print(f'  KULLANILACAK MİMARİ: {"TabPFN merkezli" if TABPFN_OK else "XGBoost fallback"}')
print(f'{"═"*50}')

## 1. Veri + 80/20 Hold-out Split

In [ ]:
train_full = pd.read_csv('train.csv')
test       = pd.read_csv('test_x.csv')
sample_sub = pd.read_csv('sample_submission.csv')

y_full = train_full[TARGET_COL].values
y_bin_full = pd.qcut(y_full, q=10, labels=False, duplicates='drop')

dev_idx, holdout_idx = train_test_split(
    np.arange(len(train_full)), test_size=HOLDOUT_RATIO,
    random_state=SEED, stratify=y_bin_full)

train_dev  = train_full.iloc[dev_idx].reset_index(drop=True)
train_hold = train_full.iloc[holdout_idx].reset_index(drop=True)
y_dev      = y_full[dev_idx]
y_hold     = y_full[holdout_idx]

print(f'Full: {train_full.shape} | Dev: {train_dev.shape} | Hold: {train_hold.shape} | Test: {test.shape}')

y_dev_bin = pd.qcut(y_dev, q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS = list(skf.split(train_dev, y_dev_bin))
test_ids = test['student_id'].values

## 2. Role-Skill Engineering

In [ ]:
ROLE_PROFILES = {
    'Cloud Engineer':        {'cloud_score':3, 'devops_score':2, 'problem_solving_score':1},
    'DevOps Engineer':       {'devops_score':3, 'cloud_score':2, 'problem_solving_score':1},
    'MLOps Engineer':        {'devops_score':2, 'machine_learning_score':2, 'cloud_score':1, 'problem_solving_score':1},
    'Frontend Developer':    {'frontend_score':3, 'coding_score':2, 'problem_solving_score':1},
    'Backend Developer':     {'backend_score':3, 'coding_score':2, 'sql_score':1, 'problem_solving_score':1},
    'Software Developer':    {'coding_score':2, 'problem_solving_score':2, 'data_structures_score':2, 'backend_score':1, 'frontend_score':1},
    'Data Scientist':        {'machine_learning_score':3, 'sql_score':2, 'problem_solving_score':2, 'data_structures_score':1},
    'AI Engineer':           {'machine_learning_score':3, 'coding_score':2, 'problem_solving_score':1, 'data_structures_score':1},
    'Data Analyst':          {'sql_score':3, 'data_structures_score':1, 'problem_solving_score':1},
    'Product Analyst':       {'sql_score':2, 'problem_solving_score':2},
    'Cybersecurity Analyst': {'devops_score':2, 'problem_solving_score':2, 'coding_score':1},
}
PRIMARY_SKILL = {r: max(w.items(), key=lambda x: x[1])[0] for r, w in ROLE_PROFILES.items()}

def add_role_features(df):
    df = df.copy()
    df['matched_skill'] = 0.0
    for role, skill in PRIMARY_SKILL.items():
        m = df['target_role'] == role
        df.loc[m, 'matched_skill'] = df.loc[m, skill].values
    df['role_composite'] = 0.0
    for role, w in ROLE_PROFILES.items():
        m = df['target_role'] == role
        if m.sum() == 0: continue
        tot = sum(w.values())
        df.loc[m, 'role_composite'] = (sum(df.loc[m, sk]*wt for sk, wt in w.items()) / tot).values
    return df

train_dev  = add_role_features(train_dev)
train_hold = add_role_features(train_hold)
train_full = add_role_features(train_full)
test       = add_role_features(test)
print('✓ Role-skill features')

## 3. NLP — BERT + TF-IDF (Bellek Güvenli, 3 Ayrı Hücre)

**v8.1 değişiklik:** Aşağıdaki 3 hücre **AYRI AYRI** çalıştırılmalı:
- 3a: BERT (kendisi ~10 sn, sonra modeli RAM'den siler)
- 3b: Word TF-IDF (~5 sn)
- 3c: Char TF-IDF (~10 sn, hafifletilmiş)

**TOPLAM: ~30 sn** (v8'deki sorunsuz hâliyle)

In [ ]:
# 3a. BERT (ayrı hücre — bellek güvenli)
BERT_OK = False
all_texts_full = pd.concat([
    train_full['mentor_feedback_text'],
    test['mentor_feedback_text']
]).fillna('').reset_index(drop=True)
n_train, n_test = len(train_full), len(test)

try:
    from sentence_transformers import SentenceTransformer
    print('Sentence-BERT yükleniyor...')
    bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=BERT_DEVICE)
    print(f'Encoding {len(all_texts_full)} text on {BERT_DEVICE}...')
    t0 = time.time()
    emb = bert_model.encode(all_texts_full.tolist(), batch_size=128, show_progress_bar=True,
                              convert_to_numpy=True, device=BERT_DEVICE)
    print(f'BERT: {emb.shape} in {time.time()-t0:.1f}s')
    pca = PCA(n_components=64, random_state=SEED)
    bert_reduced = pca.fit_transform(emb)
    print(f'PCA → 64-dim, explained: {pca.explained_variance_ratio_.sum():.3f}')
    
    bert_cols = [f'bert_{i}' for i in range(64)]
    bert_full_df = pd.DataFrame(bert_reduced[:n_train], columns=bert_cols)
    bert_test    = pd.DataFrame(bert_reduced[n_train:], columns=bert_cols)
    bert_dev  = bert_full_df.iloc[dev_idx].reset_index(drop=True)
    bert_hold = bert_full_df.iloc[holdout_idx].reset_index(drop=True)
    
    # KRİTİK: BERT modelini ve embeddings'i tamamen sil
    del bert_model, emb, bert_reduced, pca
    gc.collect()
    if BERT_DEVICE == 'cuda':
        import torch; torch.cuda.empty_cache()
    BERT_OK = True
    print('✓ BERT embeddings hazır, bellek temizlendi')
except Exception as e:
    print(f'⚠ BERT skip: {str(e)[:100]}')
    bert_dev = bert_hold = bert_test = bert_full_df = pd.DataFrame()

In [ ]:
# 3b. Word TF-IDF (hafif, hızlı)
print('Word TF-IDF çalışıyor...')
t0 = time.time()
word_tfidf = TfidfVectorizer(
    max_features=2000,        # 3000 → 2000
    ngram_range=(1,2), 
    min_df=5,                 # 3 → 5
    sublinear_tf=True, 
    analyzer='word'
)
word_m = word_tfidf.fit_transform(all_texts_full)
word_feats = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(word_m)
print(f'  Word TF-IDF + SVD: {time.time()-t0:.1f}s')
print(f'  Shape: {word_feats.shape}')

# Bellek temizle
del word_tfidf, word_m
gc.collect()
print('  ✓ Word TF-IDF tamamlandı, bellek temizlendi')

In [ ]:
# 3c. Char TF-IDF (HAFIFLETILMIŞ — Türkçe için optimum)
print('Char TF-IDF çalışıyor (hafifletilmiş parametreler)...')
t0 = time.time()
char_tfidf = TfidfVectorizer(
    max_features=1500,        # 3000 → 1500 (KRİTİK)
    ngram_range=(3, 4),       # (3, 5) → (3, 4) (KRİTİK — 5-gram patlamasını önle)
    min_df=10,                # 3 → 10 (agresif filtre)
    sublinear_tf=True, 
    analyzer='char_wb'
)
char_m = char_tfidf.fit_transform(all_texts_full)
char_feats = TruncatedSVD(n_components=30, random_state=SEED).fit_transform(char_m)
print(f'  Char TF-IDF + SVD: {time.time()-t0:.1f}s')
print(f'  Shape: {char_feats.shape}')

# Birleştir
nlp_cols = [f'nlp_w_{i}' for i in range(50)] + [f'nlp_c_{i}' for i in range(30)]
nlp_all = pd.DataFrame(np.hstack([word_feats, char_feats]), columns=nlp_cols)
nlp_all['text_len']   = all_texts_full.str.len().values
nlp_all['word_count'] = all_texts_full.str.split().str.len().values
nlp_full_df = nlp_all.iloc[:n_train].reset_index(drop=True)
nlp_test    = nlp_all.iloc[n_train:].reset_index(drop=True)
nlp_dev     = nlp_full_df.iloc[dev_idx].reset_index(drop=True)
nlp_hold    = nlp_full_df.iloc[holdout_idx].reset_index(drop=True)

# Bellek temizle
del char_tfidf, char_m, word_feats, char_feats, nlp_all
gc.collect()
print(f'✓ TF-IDF tamamlandı: {nlp_dev.shape[1]} feature, bellek temizlendi')

## 4. Target Encoding + Feature Engineering

In [ ]:
def kfold_te_dev(df_dev, df_hold, df_test, col, y_dev, splits, smoothing=20):
    gm = y_dev.mean()
    dev_t = df_dev.copy(); dev_t['__y__'] = y_dev
    oof = np.zeros(len(df_dev))
    for tr_idx, val_idx in splits:
        tr = dev_t.iloc[tr_idx]
        agg = tr.groupby(col)['__y__'].agg(['mean','count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = dev_t.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    agg = dev_t.groupby(col)['__y__'].agg(['mean','count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    return oof, df_hold[col].map(agg['s']).fillna(gm).values, df_test[col].map(agg['s']).fillna(gm).values

def kfold_te_full(df_full, df_test, col, y_full, splits, smoothing=20):
    gm = y_full.mean()
    full_t = df_full.copy(); full_t['__y__'] = y_full
    oof = np.zeros(len(df_full))
    for tr_idx, val_idx in splits:
        tr = full_t.iloc[tr_idx]
        agg = tr.groupby(col)['__y__'].agg(['mean','count'])
        agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
        val = full_t.iloc[val_idx]
        oof[val_idx] = val[col].map(agg['s']).fillna(gm).values
    agg = full_t.groupby(col)['__y__'].agg(['mean','count'])
    agg['s'] = (agg['mean']*agg['count'] + gm*smoothing)/(agg['count']+smoothing)
    return oof, df_test[col].map(agg['s']).fillna(gm).values

te_dev, te_hold, te_test_p1 = {}, {}, {}
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, ho, te = kfold_te_dev(train_dev, train_hold, test, col, y_dev, FOLD_SPLITS, 20)
    te_dev[f'te_{col}'] = oo; te_hold[f'te_{col}'] = ho; te_test_p1[f'te_{col}'] = te

for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_dev[combo]  = train_dev[c1].astype(str)+'|'+train_dev[c2].astype(str)
    train_hold[combo] = train_hold[c1].astype(str)+'|'+train_hold[c2].astype(str)
    test[combo]       = test[c1].astype(str)+'|'+test[c2].astype(str)
    oo, ho, te = kfold_te_dev(train_dev, train_hold, test, combo, y_dev, FOLD_SPLITS, 30)
    te_dev[f'te_{combo}'] = oo; te_hold[f'te_{combo}'] = ho; te_test_p1[f'te_{combo}'] = te
    train_dev.drop(columns=[combo], inplace=True)
    train_hold.drop(columns=[combo], inplace=True)
    test.drop(columns=[combo], inplace=True)

train_te_p1 = pd.DataFrame(te_dev)
hold_te_p1  = pd.DataFrame(te_hold)
test_te_p1  = pd.DataFrame(te_test_p1)
print(f'✓ Target encoding: {train_te_p1.shape[1]} features')

In [ ]:
TIER_MAP  = {'Tier 1':4,'Tier 2':3,'Tier 3':2,'Tier 4':1}
TECH_COLS = ['coding_score','problem_solving_score','data_structures_score',
             'sql_score','machine_learning_score','backend_score','frontend_score',
             'cloud_score','devops_score']
SOFT_COLS = ['communication_score','teamwork_score','leadership_score','presentation_score']
CAT_COLS  = ['department','target_role','hobby','preferred_social_media_platform']

def derive_features(df):
    df = df.copy()
    df['tech_mean']  = df[TECH_COLS].mean(axis=1)
    df['tech_max']   = df[TECH_COLS].max(axis=1)
    df['tech_min']   = df[TECH_COLS].min(axis=1)
    df['tech_std']   = df[TECH_COLS].std(axis=1)
    df['tech_top3']  = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)
    df['tech_bot3']  = df[TECH_COLS].apply(lambda r: r.nsmallest(3).mean(), axis=1)
    df['soft_mean'] = df[SOFT_COLS].mean(axis=1)
    df['soft_std']  = df[SOFT_COLS].std(axis=1)
    df['matched_x_proj']        = df['matched_skill'] * df['project_quality_score']
    df['matched_x_interview']   = df['matched_skill'] * df['technical_interview_score']
    df['role_comp_x_proj']      = df['role_composite'] * df['project_quality_score']
    df['role_comp_minus_avg']   = df['role_composite'] - df['tech_mean']
    df['matched_minus_tech']    = df['matched_skill'] - df['tech_mean']
    df['interview_composite']   = df['technical_interview_score']*0.6 + df['hr_interview_score']*0.4
    df['experience_score']      = (df['internship_count']*3 + df['real_client_project_count']*2 + df['freelance_project_count'])
    df['github_score']          = (df['github_repo_count'] * np.log1p(df['github_avg_stars']+1) + df['open_source_contribution_count'])
    df['profile_composite']     = (df['portfolio_score']+df['linkedin_profile_score']+df['cv_quality_score'])/3
    df['university_tier_num']   = df['university_tier'].map(TIER_MAP)
    df['cgpa_x_attendance']     = df['cgpa'] * df['attendance_rate']
    df['proj_x_interview']      = df['project_quality_score'] * df['technical_interview_score']
    df['overall_score'] = (df['tech_mean']*0.25 + df['soft_mean']*0.15 +
                            df['project_quality_score']*0.25 + df['profile_composite']*0.10 +
                            df['interview_composite']*0.15 + df['matched_skill']*0.10)
    return df

def prepare_features(df_train, df_test, fill_medians=None, label_encoders=None):
    df_train = df_train.copy(); df_test = df_test.copy()
    fill_cols = ['english_exam_score','internship_duration_months','portfolio_score',
                 'github_avg_stars','open_source_contribution_count',
                 'linkedin_profile_score','hr_interview_score']
    if fill_medians is None:
        fill_medians = {c: df_train[c].median() for c in fill_cols}
    for c in fill_cols:
        df_train[c] = df_train[c].fillna(fill_medians[c])
        df_test[c]  = df_test[c].fillna(fill_medians[c])
    df_train = derive_features(df_train)
    df_test  = derive_features(df_test)
    df_train_cat = df_train.copy()
    df_test_cat  = df_test.copy()
    if label_encoders is None:
        label_encoders = {}
        for col in CAT_COLS + ['university_tier']:
            le = LabelEncoder()
            le.fit(pd.concat([df_train[col], df_test[col]]).astype(str))
            label_encoders[col] = le
    for col, le in label_encoders.items():
        df_train[col] = le.transform(df_train[col].astype(str))
        df_test[col]  = le.transform(df_test[col].astype(str))
    drop_cols = ['student_id','mentor_feedback_text']
    if TARGET_COL in df_train.columns: drop_cols.append(TARGET_COL)
    df_train = df_train.drop(columns=drop_cols, errors='ignore')
    df_test  = df_test.drop(columns=drop_cols, errors='ignore')
    df_train_cat = df_train_cat.drop(columns=drop_cols, errors='ignore')
    df_test_cat  = df_test_cat.drop(columns=drop_cols, errors='ignore')
    return df_train, df_test, df_train_cat, df_test_cat, fill_medians, label_encoders

X_dev_base, X_test_base_p1, X_dev_cat, X_test_cat_p1, dev_medians, dev_les = prepare_features(train_dev, test)
X_hold_base, _, X_hold_cat, _, _, _ = prepare_features(train_hold, test, fill_medians=dev_medians, label_encoders=dev_les)

def merge_all(base, nlp, te, bert):
    parts = [base.reset_index(drop=True), nlp.reset_index(drop=True), te.reset_index(drop=True)]
    if BERT_OK and len(bert) > 0:
        parts.append(bert.reset_index(drop=True))
    return pd.concat(parts, axis=1)

X_dev      = merge_all(X_dev_base, nlp_dev, train_te_p1, bert_dev)
X_hold     = merge_all(X_hold_base, nlp_hold, hold_te_p1, bert_hold)
X_test_p1  = merge_all(X_test_base_p1, nlp_test, test_te_p1, bert_test)
X_dev_cat  = merge_all(X_dev_cat, nlp_dev, train_te_p1, bert_dev)
X_hold_cat = merge_all(X_hold_cat, nlp_hold, hold_te_p1, bert_hold)
X_test_cat_p1 = merge_all(X_test_cat_p1, nlp_test, test_te_p1, bert_test)

col_medians = X_dev.median(numeric_only=True)
X_dev      = X_dev.fillna(col_medians)
X_hold     = X_hold.fillna(col_medians)
X_test_p1  = X_test_p1.fillna(col_medians)
for c in X_dev_cat.columns:
    if pd.api.types.is_numeric_dtype(X_dev_cat[c]) and X_dev_cat[c].isna().any():
        X_dev_cat[c]  = X_dev_cat[c].fillna(col_medians.get(c, 0))
        X_hold_cat[c] = X_hold_cat[c].fillna(col_medians.get(c, 0))
        X_test_cat_p1[c] = X_test_cat_p1[c].fillna(col_medians.get(c, 0))

cat_features_idx = [X_dev_cat.columns.get_loc(c) for c in CAT_COLS + ['university_tier']]
print(f'✓ Final features: Dev {X_dev.shape}, Hold {X_hold.shape}, Test {X_test_p1.shape}')

## 5. Feature Selection (XGB importance)

In [ ]:
print('Feature importance hesaplanıyor...')
quick = xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=SEED, **xgb_gpu())
quick.fit(X_dev, y_dev)
imps = pd.Series(quick.feature_importances_, index=X_dev.columns).sort_values(ascending=False)

all_features_keep = imps[imps > imps.quantile(0.15)].index.tolist()
# TabPFN için: en önemli 100 feature (model maksimum 500 destekliyor)
top_100_features = imps.head(min(100, len(imps))).index.tolist()

X_dev_sel = X_dev[all_features_keep]
X_hold_sel = X_hold[all_features_keep]
X_test_sel_p1 = X_test_p1[all_features_keep]
X_dev_cat_sel = X_dev_cat[all_features_keep]
X_hold_cat_sel = X_hold_cat[all_features_keep]
X_test_cat_sel_p1 = X_test_cat_p1[all_features_keep]
cat_idx_sel = [X_dev_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_dev_cat_sel.columns]

# TabPFN için ayrı subset (top 100)
X_dev_tp = X_dev[top_100_features]
X_hold_tp = X_hold[top_100_features]
X_test_tp_p1 = X_test_p1[top_100_features]

print(f'  Trees: {len(all_features_keep)} feature')
print(f'  TabPFN: {len(top_100_features)} feature (max 500 destekliyor)')

## 6. FLAML Hyperparameter Tuning (LightGBM + CatBoost)

In [ ]:
opt_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
OPT_SPLITS = list(opt_skf.split(X_dev_sel, y_dev_bin))

# LightGBM FLAML
def lgb_eval(config):
    rmses = []
    for tr, va in OPT_SPLITS:
        m = lgb.LGBMRegressor(**config, random_state=SEED, verbose=-1,
                                n_estimators=2000, **lgb_gpu())
        m.fit(X_dev_sel.iloc[tr], y_dev[tr],
              eval_set=[(X_dev_sel.iloc[va], y_dev[va])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_sel.iloc[va]))))
    return {'rmse': np.mean(rmses)}

lgb_space = {
    'learning_rate': flaml_tune.loguniform(0.005, 0.1),
    'num_leaves': flaml_tune.randint(31, 400),
    'max_depth': flaml_tune.randint(4, 12),
    'subsample': flaml_tune.uniform(0.6, 1.0),
    'colsample_bytree': flaml_tune.uniform(0.5, 1.0),
    'min_child_samples': flaml_tune.randint(5, 100),
    'reg_alpha': flaml_tune.loguniform(1e-3, 10),
    'reg_lambda': flaml_tune.loguniform(1e-3, 10),
}

print('LightGBM FLAML tuning (~5 dakika)...')
t0 = time.time()
lgb_analysis = flaml_tune.run(
    lgb_eval, config=lgb_space, metric='rmse', mode='min',
    num_samples=40, time_budget_s=300, verbose=0
)
lgb_best_params = lgb_analysis.best_config
print(f'LGB best RMSE: {lgb_analysis.best_result["rmse"]:.4f}  ({time.time()-t0:.0f}s)')

In [ ]:
# CatBoost FLAML
def cat_eval(config):
    rmses = []
    for tr, va in OPT_SPLITS:
        m = CatBoostRegressor(**config, random_seed=SEED, verbose=0, loss_function='RMSE',
                                iterations=2000, early_stopping_rounds=50, **cat_gpu())
        m.fit(X_dev_cat_sel.iloc[tr], y_dev[tr],
              eval_set=(X_dev_cat_sel.iloc[va], y_dev[va]),
              cat_features=cat_idx_sel, verbose=0)
        rmses.append(np.sqrt(mean_squared_error(y_dev[va], m.predict(X_dev_cat_sel.iloc[va]))))
    return {'rmse': np.mean(rmses)}

cat_space = {
    'learning_rate': flaml_tune.loguniform(0.005, 0.1),
    'depth': flaml_tune.randint(4, 10),
    'l2_leaf_reg': flaml_tune.uniform(1, 15),
    'min_data_in_leaf': flaml_tune.randint(1, 100),
    'random_strength': flaml_tune.uniform(0.5, 5),
    'bagging_temperature': flaml_tune.uniform(0, 1),
}

print('CatBoost FLAML tuning (~5 dakika)...')
t0 = time.time()
cat_analysis = flaml_tune.run(
    cat_eval, config=cat_space, metric='rmse', mode='min',
    num_samples=30, time_budget_s=300, verbose=0
)
cat_best_params = cat_analysis.best_config
print(f'CAT best RMSE: {cat_analysis.best_result["rmse"]:.4f}  ({time.time()-t0:.0f}s)')

## 7. Stacking — TabPFN + LightGBM + CatBoost

In [ ]:
def stack_p1(factory, splits, X_tr, X_ho, X_te, y_tr, n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    oof = np.zeros(len(y_tr)); ho_p = np.zeros(len(X_ho)); te_p = np.zeros(len(X_te))
    per_fold_mse = []
    for seed in range(n_seeds):
        fold_ho = np.zeros(len(X_ho)); fold_te = np.zeros(len(X_te))
        for fold_i, (tr_idx, val_idx) in enumerate(splits):
            m = factory(seed)
            if kind == 'tabpfn':
                X_tr_np = X_tr.iloc[tr_idx].values if hasattr(X_tr, 'iloc') else X_tr[tr_idx]
                X_va_np = X_tr.iloc[val_idx].values if hasattr(X_tr, 'iloc') else X_tr[val_idx]
                X_ho_np = X_ho.values if hasattr(X_ho, 'values') else X_ho
                X_te_np = X_te.values if hasattr(X_te, 'values') else X_te
                m.fit(X_tr_np, y_tr[tr_idx])
                val_pred = m.predict(X_va_np)
                ho_pred = m.predict(X_ho_np)
                te_pred = m.predict(X_te_np)
            elif kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
                val_pred = m.predict(X_tr.iloc[val_idx])
                ho_pred = m.predict(X_ho); te_pred = m.predict(X_te)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
                val_pred = m.predict(X_tr.iloc[val_idx])
                ho_pred = m.predict(X_ho); te_pred = m.predict(X_te)
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
                val_pred = m.predict(X_tr.iloc[val_idx])
                ho_pred = m.predict(X_ho); te_pred = m.predict(X_te)
            else:
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
                val_pred = m.predict(X_tr.iloc[val_idx])
                ho_pred = m.predict(X_ho); te_pred = m.predict(X_te)
            oof[val_idx] += val_pred / n_seeds
            fold_ho += ho_pred / len(splits)
            fold_te += te_pred / len(splits)
            if seed == 0:
                per_fold_mse.append(mean_squared_error(y_tr[val_idx], val_pred))
        ho_p += fold_ho / n_seeds
        te_p += fold_te / n_seeds
    return oof, ho_p, te_p, per_fold_mse

print(f'Phase 1 Stacking ({N_FOLDS}-fold × seeds × 3 model)...\n')
stability = {}

# Model factories
lgb_factory = lambda s: lgb.LGBMRegressor(**lgb_best_params, random_state=SEED+s, **lgb_gpu(),
                                            n_estimators=3000, verbose=-1)
cat_factory = lambda s: CatBoostRegressor(**cat_best_params, random_seed=SEED+s, **cat_gpu(),
                                            iterations=3000, verbose=0, loss_function='RMSE',
                                            early_stopping_rounds=50)

# TabPFN factory (no tuning needed, internal ensemble already)
if TABPFN_OK:
    from tabpfn import TabPFNRegressor
    tabpfn_factory = lambda s: TabPFNRegressor(
        device=TABPFN_DEVICE, n_estimators=4, random_state=SEED+s,
        ignore_pretraining_limits=True
    )
else:
    # Fallback to XGBoost
    print('⚠ TabPFN devre dışı → XGBoost fallback kullanılıyor')
    tabpfn_factory = lambda s: xgb.XGBRegressor(
        n_estimators=3000, learning_rate=0.05, max_depth=7, random_state=SEED+s,
        **xgb_gpu(), early_stopping_rounds=50,
        subsample=0.8, colsample_bytree=0.8
    )

# Run stacking
t0 = time.time()
if TABPFN_OK:
    # TabPFN: n_seeds=1 (zaten internal ensemble var)
    tp_oof, tp_ho, tp_te, tp_fmse = stack_p1(tabpfn_factory, FOLD_SPLITS, X_dev_tp, X_hold_tp, X_test_tp_p1, y_dev, n_seeds=1, kind='tabpfn')
    main_label = 'TabPFN'
else:
    tp_oof, tp_ho, tp_te, tp_fmse = stack_p1(tabpfn_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel_p1, y_dev, n_seeds=N_SEEDS, kind='xgb')
    main_label = 'XGB(fallback)'
stability[main_label] = tp_fmse
print(f'  {main_label}: OOF MSE={mean_squared_error(y_dev, tp_oof):.3f}  Hold MSE={mean_squared_error(y_hold, tp_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
lgb_oof, lgb_ho, lgb_te, lgb_fmse = stack_p1(lgb_factory, FOLD_SPLITS, X_dev_sel, X_hold_sel, X_test_sel_p1, y_dev, kind='lgb')
stability['LGB'] = lgb_fmse
print(f'  LGB     : OOF MSE={mean_squared_error(y_dev, lgb_oof):.3f}  Hold MSE={mean_squared_error(y_hold, lgb_ho):.3f}  ({time.time()-t0:.0f}s)')

t0 = time.time()
cat_oof, cat_ho, cat_te, cat_fmse = stack_p1(cat_factory, FOLD_SPLITS, X_dev_cat_sel, X_hold_cat_sel, X_test_cat_sel_p1, y_dev, kind='cat', cat_idx=cat_idx_sel)
stability['CAT'] = cat_fmse
print(f'  CAT     : OOF MSE={mean_squared_error(y_dev, cat_oof):.3f}  Hold MSE={mean_squared_error(y_hold, cat_ho):.3f}  ({time.time()-t0:.0f}s)')

oof_matrix     = np.column_stack([tp_oof, lgb_oof, cat_oof])
holdout_matrix = np.column_stack([tp_ho, lgb_ho, cat_ho])
test_matrix_p1 = np.column_stack([tp_te, lgb_te, cat_te])
model_names    = [main_label, 'LGB', 'CAT']
n_models       = 3

## 8. Stability + Weight Optimization + Holdout Validation

In [ ]:
print('Per-fold MSE stability:\n')
for name, fmse in stability.items():
    mn = np.mean(fmse); sd = np.std(fmse); cv = sd/mn*100
    flag = '✓ STABLE' if cv < 5 else '⚠️ UNSTABLE' if cv < 10 else '🚨 VERY UNSTABLE'
    print(f'  {name:<15} mean={mn:6.3f}  std={sd:5.3f}  CV={cv:5.2f}%  {flag}')

# Weight optimization
def weighted_rmse(w, preds, y_true):
    return np.sqrt(mean_squared_error(y_true, preds @ w))

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.0, 1.0)] * n_models
best_loss, best_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_loss: best_loss, best_w = res.fun, res.x

best_ridge_alpha, best_ridge_meta_loss = None, float('inf')
meta_kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for alpha in [0.01, 0.1, 1, 5, 10, 50, 100]:
    fold_mses = []
    for tr_idx, val_idx in meta_kf.split(oof_matrix, y_dev_bin):
        meta = Ridge(alpha=alpha); meta.fit(oof_matrix[tr_idx], y_dev[tr_idx])
        fold_mses.append(np.sqrt(mean_squared_error(y_dev[val_idx], meta.predict(oof_matrix[val_idx]))))
    avg = np.mean(fold_mses)
    if avg < best_ridge_meta_loss: best_ridge_meta_loss, best_ridge_alpha = avg, alpha

ridge_meta = Ridge(alpha=best_ridge_alpha); ridge_meta.fit(oof_matrix, y_dev)

slsqp_ho_mse = mean_squared_error(y_hold, holdout_matrix @ best_w)
ridge_ho_mse = mean_squared_error(y_hold, ridge_meta.predict(holdout_matrix))
blend_ho_mse = mean_squared_error(y_hold, 0.5*(holdout_matrix @ best_w) + 0.5*ridge_meta.predict(holdout_matrix))

print('\nENSEMBLE — Holdout MSE:')
print(f'  SLSQP     : {slsqp_ho_mse:.3f}')
print(f'  Ridge     : {ridge_ho_mse:.3f}')
print(f'  Blend     : {blend_ho_mse:.3f}')
for n, w in zip(model_names, best_w):
    print(f'  weight {n:<15}: {w:.4f}')

options = [
    ('SLSQP', slsqp_ho_mse, lambda M: M @ best_w),
    ('Ridge', ridge_ho_mse, lambda M: ridge_meta.predict(M)),
    ('Blend', blend_ho_mse, lambda M: 0.5*(M @ best_w) + 0.5*ridge_meta.predict(M))
]
options.sort(key=lambda x: x[1])
best_method_name, best_method_holdout, best_method_fn = options[0]
print(f'\n🏆 En iyi: {best_method_name}  (Holdout MSE = {best_method_holdout:.3f})')

## 9. Akıllı Pseudo-Labeling A/B Test

In [ ]:
test_stds = test_matrix_p1.std(axis=1)
PSEUDO_FRAC = 0.10
n_pseudo = int(len(test) * PSEUDO_FRAC)
conf_idx = np.argsort(test_stds)[:n_pseudo]
pseudo_labels = np.clip(best_method_fn(test_matrix_p1)[conf_idx], 0, 100)

print(f'Pseudo: {n_pseudo} test örneği eklenecek')

X_dev_sel_aug    = pd.concat([X_dev_sel, X_test_sel_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_cat_aug    = pd.concat([X_dev_cat_sel, X_test_cat_sel_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
X_dev_tp_aug     = pd.concat([X_dev_tp, X_test_tp_p1.iloc[conf_idx]], axis=0).reset_index(drop=True)
y_dev_aug        = np.concatenate([y_dev, pseudo_labels])
y_dev_aug_bin    = pd.qcut(y_dev_aug, q=10, labels=False, duplicates='drop')
FOLD_SPLITS_AUG  = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_dev_sel_aug, y_dev_aug_bin))
n_orig = len(y_dev)

t0 = time.time()
if TABPFN_OK:
    tp_oof2, tp_ho2, tp_te2, _ = stack_p1(tabpfn_factory, FOLD_SPLITS_AUG, X_dev_tp_aug, X_hold_tp, X_test_tp_p1, y_dev_aug, n_seeds=1, kind='tabpfn')
else:
    tp_oof2, tp_ho2, tp_te2, _ = stack_p1(tabpfn_factory, FOLD_SPLITS_AUG, X_dev_sel_aug, X_hold_sel, X_test_sel_p1, y_dev_aug, n_seeds=N_SEEDS, kind='xgb')
lgb_oof2, lgb_ho2, lgb_te2, _ = stack_p1(lgb_factory, FOLD_SPLITS_AUG, X_dev_sel_aug, X_hold_sel, X_test_sel_p1, y_dev_aug, kind='lgb')
cat_oof2, cat_ho2, cat_te2, _ = stack_p1(cat_factory, FOLD_SPLITS_AUG, X_dev_cat_aug, X_hold_cat_sel, X_test_cat_sel_p1, y_dev_aug, kind='cat', cat_idx=cat_idx_sel)
print(f'Pseudo stacking: {time.time()-t0:.0f}s')

oof_matrix_aug    = np.column_stack([tp_oof2[:n_orig], lgb_oof2[:n_orig], cat_oof2[:n_orig]])
holdout_matrix_aug= np.column_stack([tp_ho2, lgb_ho2, cat_ho2])
test_matrix_aug   = np.column_stack([tp_te2, lgb_te2, cat_te2])

best_aug_loss, best_aug_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_matrix_aug, y_dev),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_aug_loss: best_aug_loss, best_aug_w = res.fun, res.x

ridge_meta_aug = Ridge(alpha=best_ridge_alpha); ridge_meta_aug.fit(oof_matrix_aug, y_dev)
aug_slsqp_ho = mean_squared_error(y_hold, holdout_matrix_aug @ best_aug_w)
aug_ridge_ho = mean_squared_error(y_hold, ridge_meta_aug.predict(holdout_matrix_aug))
aug_blend_ho = mean_squared_error(y_hold, 0.5*(holdout_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(holdout_matrix_aug))

all_options = [
    ('Orig+SLSQP', slsqp_ho_mse, lambda: best_method_fn(test_matrix_p1)),
    ('Orig+Ridge', ridge_ho_mse, lambda: ridge_meta.predict(test_matrix_p1)),
    ('Orig+Blend', blend_ho_mse, lambda: 0.5*(test_matrix_p1 @ best_w) + 0.5*ridge_meta.predict(test_matrix_p1)),
    ('Pseudo+SLSQP', aug_slsqp_ho, lambda: test_matrix_aug @ best_aug_w),
    ('Pseudo+Ridge', aug_ridge_ho, lambda: ridge_meta_aug.predict(test_matrix_aug)),
    ('Pseudo+Blend', aug_blend_ho, lambda: 0.5*(test_matrix_aug @ best_aug_w) + 0.5*ridge_meta_aug.predict(test_matrix_aug)),
]

best_orig_mse = min(slsqp_ho_mse, ridge_ho_mse, blend_ho_mse)
best_pseudo_mse = min(aug_slsqp_ho, aug_ridge_ho, aug_blend_ho)

print('\nA/B TEST:')
for name, mse, _ in all_options:
    print(f'  {name:<18}: {mse:.4f}')

if best_pseudo_mse < best_orig_mse:
    USE_PSEUDO = True
    print(f'\n✓ Pseudo iyileştirdi ({best_orig_mse-best_pseudo_mse:+.4f}) → KULLANILACAK')
else:
    USE_PSEUDO = False
    print(f'\n✗ Pseudo kötüleştirdi ({best_pseudo_mse-best_orig_mse:+.4f}) → ATLANIYOR')
    all_options = [o for o in all_options if 'Pseudo' not in o[0]]

all_options.sort(key=lambda x: x[1])
winner_name, winner_holdout_mse, winner_test_fn = all_options[0]
print(f'🏆 PHASE 1 KAZANAN: {winner_name}  (Holdout MSE = {winner_holdout_mse:.4f})')
final_test_phase1 = np.clip(winner_test_fn(), 0, 100)

## 10. Phase 4 — Full Retrain (10000)

In [ ]:
print('Phase 4: full 10000 ile yeniden eğitim...\n')

X_full_base, X_test_base_p4, X_full_cat, X_test_cat_p4, _, _ = prepare_features(train_full, test)
skf_full = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLD_SPLITS_FULL = list(skf_full.split(train_full, y_bin_full))

te_full, te_test_p4 = {}, {}
for col in ['department','target_role','hobby','preferred_social_media_platform','university_tier']:
    oo, te = kfold_te_full(train_full, test, col, y_full, FOLD_SPLITS_FULL, 20)
    te_full[f'te_{col}'] = oo; te_test_p4[f'te_{col}'] = te
for c1, c2 in [('department','target_role'), ('university_tier','target_role'), ('target_role','hobby')]:
    combo = f'{c1}_x_{c2}'
    train_full[combo] = train_full[c1].astype(str)+'|'+train_full[c2].astype(str)
    test_t = test.copy(); test_t[combo] = test_t[c1].astype(str)+'|'+test_t[c2].astype(str)
    oo, te = kfold_te_full(train_full, test_t, combo, y_full, FOLD_SPLITS_FULL, 30)
    te_full[f'te_{combo}'] = oo; te_test_p4[f'te_{combo}'] = te
    train_full.drop(columns=[combo], inplace=True)

train_te_p4 = pd.DataFrame(te_full)
test_te_p4 = pd.DataFrame(te_test_p4)

X_full = merge_all(X_full_base, nlp_full_df, train_te_p4, bert_full_df if BERT_OK else pd.DataFrame())
X_test_p4 = merge_all(X_test_base_p4, nlp_test, test_te_p4, bert_test if BERT_OK else pd.DataFrame())
X_full_cat = merge_all(X_full_cat, nlp_full_df, train_te_p4, bert_full_df if BERT_OK else pd.DataFrame())
X_test_cat_p4 = merge_all(X_test_cat_p4, nlp_test, test_te_p4, bert_test if BERT_OK else pd.DataFrame())

full_medians = X_full.median(numeric_only=True)
X_full = X_full.fillna(full_medians)
X_test_p4 = X_test_p4.fillna(full_medians)
for c in X_full_cat.columns:
    if pd.api.types.is_numeric_dtype(X_full_cat[c]) and X_full_cat[c].isna().any():
        X_full_cat[c] = X_full_cat[c].fillna(full_medians.get(c, 0))
        X_test_cat_p4[c] = X_test_cat_p4[c].fillna(full_medians.get(c, 0))

use_cols = [c for c in all_features_keep if c in X_full.columns]
use_tp = [c for c in top_100_features if c in X_full.columns]
X_full_sel = X_full[use_cols]
X_test_sel_p4 = X_test_p4[use_cols]
X_full_cat_sel = X_full_cat[use_cols]
X_test_cat_sel_p4 = X_test_cat_p4[use_cols]
X_full_tp = X_full[use_tp]
X_test_tp_p4 = X_test_p4[use_tp]
cat_idx_full = [X_full_cat_sel.columns.get_loc(c) for c in CAT_COLS + ['university_tier'] if c in X_full_cat_sel.columns]

In [ ]:
def stack_p4(factory, splits, X_tr, X_te, y_tr, n_seeds=N_SEEDS, kind='plain', cat_idx=None):
    oof = np.zeros(len(y_tr)); te_p = np.zeros(len(X_te))
    for seed in range(n_seeds):
        fold_te = np.zeros(len(X_te))
        for tr_idx, val_idx in splits:
            m = factory(seed)
            if kind == 'tabpfn':
                X_tr_np = X_tr.iloc[tr_idx].values; X_va_np = X_tr.iloc[val_idx].values; X_te_np = X_te.values
                m.fit(X_tr_np, y_tr[tr_idx])
                oof[val_idx] += m.predict(X_va_np) / n_seeds
                fold_te += m.predict(X_te_np) / len(splits)
            elif kind == 'xgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])], verbose=False)
                oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
                fold_te += m.predict(X_te) / len(splits)
            elif kind == 'lgb':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=[(X_tr.iloc[val_idx], y_tr[val_idx])],
                      callbacks=[lgb.early_stopping(50, verbose=False)])
                oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
                fold_te += m.predict(X_te) / len(splits)
            elif kind == 'cat':
                m.fit(X_tr.iloc[tr_idx], y_tr[tr_idx], eval_set=(X_tr.iloc[val_idx], y_tr[val_idx]),
                      cat_features=cat_idx, verbose=0)
                oof[val_idx] += m.predict(X_tr.iloc[val_idx]) / n_seeds
                fold_te += m.predict(X_te) / len(splits)
        te_p += fold_te / n_seeds
    return oof, te_p

t0 = time.time()
if TABPFN_OK:
    f_tp_oof, f_tp_te = stack_p4(tabpfn_factory, FOLD_SPLITS_FULL, X_full_tp, X_test_tp_p4, y_full, n_seeds=1, kind='tabpfn')
else:
    f_tp_oof, f_tp_te = stack_p4(tabpfn_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_sel_p4, y_full, kind='xgb')
print(f'  {main_label} done ({time.time()-t0:.0f}s)')
t0 = time.time()
f_lgb_oof, f_lgb_te = stack_p4(lgb_factory, FOLD_SPLITS_FULL, X_full_sel, X_test_sel_p4, y_full, kind='lgb')
print(f'  LGB done ({time.time()-t0:.0f}s)')
t0 = time.time()
f_cat_oof, f_cat_te = stack_p4(cat_factory, FOLD_SPLITS_FULL, X_full_cat_sel, X_test_cat_sel_p4, y_full, kind='cat', cat_idx=cat_idx_full)
print(f'  CAT done ({time.time()-t0:.0f}s)')

oof_full_matrix = np.column_stack([f_tp_oof, f_lgb_oof, f_cat_oof])
test_full_matrix = np.column_stack([f_tp_te, f_lgb_te, f_cat_te])

best_full_loss, best_full_w = float('inf'), None
for trial in range(100):
    start = np.random.dirichlet(np.ones(n_models)) if trial > 0 else np.ones(n_models)/n_models
    res = minimize(weighted_rmse, start, args=(oof_full_matrix, y_full),
                    method='SLSQP', constraints=constraints, bounds=bounds,
                    options={'maxiter': 1000, 'ftol': 1e-10})
    if res.fun < best_full_loss: best_full_loss, best_full_w = res.fun, res.x

ridge_meta_full = Ridge(alpha=best_ridge_alpha); ridge_meta_full.fit(oof_full_matrix, y_full)
p4_slsqp_oof = oof_full_matrix @ best_full_w
p4_ridge_oof = ridge_meta_full.predict(oof_full_matrix)
p4_blend_oof = 0.5*p4_slsqp_oof + 0.5*p4_ridge_oof

p4_options = [
    ('Phase4 SLSQP', mean_squared_error(y_full, p4_slsqp_oof), test_full_matrix @ best_full_w),
    ('Phase4 Ridge', mean_squared_error(y_full, p4_ridge_oof), ridge_meta_full.predict(test_full_matrix)),
    ('Phase4 Blend', mean_squared_error(y_full, p4_blend_oof), 0.5*(test_full_matrix @ best_full_w) + 0.5*ridge_meta_full.predict(test_full_matrix))
]
p4_options.sort(key=lambda x: x[1])
p4_winner_name, p4_winner_mse, p4_winner_pred = p4_options[0]
final_test_phase4 = np.clip(p4_winner_pred, 0, 100)

print(f'\nPhase 4 OOF MSE: {p4_winner_mse:.4f}  [{p4_winner_name}]')
for n, w in zip(model_names, best_full_w):
    print(f'  {n}: {w:.4f}')

## 11. 🏁 Final Submission

In [ ]:
# 70% Phase4 + 30% Phase1
final_test = 0.7 * final_test_phase4 + 0.3 * final_test_phase1
final_test = np.clip(final_test, 0, 100)

submission = pd.DataFrame({
    'student_id'         : test_ids,
    'career_success_score': final_test
})
submission.to_csv('submission.csv', index=False)
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase1}).to_csv('submission_phase1.csv', index=False)
pd.DataFrame({'student_id': test_ids, 'career_success_score': final_test_phase4}).to_csv('submission_phase4.csv', index=False)

print('═'*60)
print('  🏆 v8 FINAL RESULTS')
print('═'*60)
print(f'  Ana model              : {main_label}')
print(f'  Phase 1 Holdout MSE    : {winner_holdout_mse:.4f}')
print(f'  Phase 4 OOF MSE        : {p4_winner_mse:.4f}')
print(f'  Pseudo-labeling        : {"KULLANILDI" if USE_PSEUDO else "ATLANDI"}')
print(f'  Final blend            : 70% Phase4 + 30% Phase1')
print('-'*60)
print(f'  Leaderboard referans:')
print(f'    1. sıra: 80.99   |   10. sıra: 82.46   |   Sen (v1): 89.54')
best_est = min(winner_holdout_mse, p4_winner_mse)
print(f'    v8 beklenen: ~{best_est:.2f}')
print('═'*60)
print(f'\n3 submission dosyası üretildi:')
print(f'  ⭐ submission.csv (70/30 blend - İLK BU)')
print(f'     submission_phase1.csv (dev, holdout-validated)')
print(f'     submission_phase4.csv (full 10000 retrain)')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

dev_mses = [mean_squared_error(y_dev, oof_matrix[:,i]) for i in range(n_models)]
hold_mses = [mean_squared_error(y_hold, holdout_matrix[:,i]) for i in range(n_models)]
x = np.arange(len(model_names))
axes[0,0].bar(x-0.2, dev_mses, 0.4, label='Dev OOF', color='steelblue')
axes[0,0].bar(x+0.2, hold_mses, 0.4, label='Holdout', color='coral')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(model_names)
axes[0,0].legend(); axes[0,0].set_title('OOF vs Holdout MSE'); axes[0,0].set_ylabel('MSE')

axes[0,1].hist(y_full, bins=50, alpha=0.6, color='steelblue', label='Train', edgecolor='white')
axes[0,1].hist(final_test, bins=50, alpha=0.6, color='coral', label='Test pred', edgecolor='white')
axes[0,1].set_title('Dağılım'); axes[0,1].legend()

fold_data = [{'Model': name, 'MSE': v} for name, fmse in stability.items() for v in fmse]
sns.boxplot(data=pd.DataFrame(fold_data), x='Model', y='MSE', ax=axes[1,0])
axes[1,0].set_title('Per-Fold MSE Stability')

axes[1,1].pie(best_full_w, labels=model_names, autopct='%1.1f%%')
axes[1,1].set_title('Phase 4 Ensemble Weights')

plt.tight_layout(); plt.show()

---
## 📋 v8 Mimari Özet

| Bileşen | Açıklama |
|---------|----------|
| **Ana Model** | TabPFN-v3 (Nature 2025, foundation model, NO TUNING) |
| **Destek Modelleri** | LightGBM + CatBoost (FLAML tuned) |
| **Hyperparam Opt.** | FLAML (Optuna'dan 2-3x hızlı) |
| **NLP** | Sentence-BERT (multilingual) + TF-IDF (word + char) |
| **Target Encoding** | Single + Cross-categorical (sızıntısız) |
| **Validation** | 80/20 hold-out + 10-fold StratifiedKFold |
| **Stacking** | Multi-seed + meta-learning + holdout validation |
| **Pseudo-Labeling** | Akıllı (sadece holdout iyileşirse) |
| **Final** | 70% Phase4 + 30% Phase1 |
| **Fallback** | TabPFN auth yoksa → XGBoost otomatik devreye |

## 🎯 Beklenen Sonuç

| Versiyon | Tahmini MSE | Sıralama |
|----------|-------------|----------|
| v6 (yapılmış) | 83-85 | top 10-20 |
| **v8 (TabPFN aktif)** | **78-82** | **🥇 top 1-5** |
| v8 (fallback) | 82-84 | top 5-15 |

## ⏱️ Süre (A100 GPU)
- BERT + features: ~10 dk
- FLAML tuning (LGB + CAT): ~10 dk
- Stacking + Pseudo + Phase 4: ~30-45 dk
- **TOPLAM: ~50-65 dk**